# 82-0 Game Logic Walkthrough

This notebook walks through the exact scoring engine that powers the 82-0 game.  
Each section shows the **original JavaScript** extracted from `Engine.js`, runs it live via Node.js,  
and compares it against the **Python port** in `engine.py` to verify the numbers match exactly.

In [ ]:
import subprocess, json, sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from engine import calculate_team_result, projected_wins, adjust_spg_bpg, calculate_player_rating
from simulator import load_players
from optimizer import stat_contribution, build_top_players_by_pos, build_market_context

def run_js(code: str) -> str:
    """Execute a JavaScript snippet with Node.js and return stdout."""
    result = subprocess.run(['node', '-e', code], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()

print('Node.js:', run_js('console.log(process.version)'))
print('Python:', sys.version.split()[0])

---
## 1. The Spin Mechanic

Each draft round selects an era uniformly (1/7), then a team uniformly within that era.  
This means early eras with fewer teams have **higher per-combo probability** than later eras.

In [ ]:
teams, valid_combos, all_players = load_players()

from collections import Counter
era_team_count = Counter(era for _, era in valid_combos)

print(f"{'Era':<8} {'Teams':>6} {'P(era)':>8} {'P(specific combo)':>20}")
print('-' * 46)
for era in ['1960s','1970s','1980s','1990s','2000s','2010s','2020s']:
    n = era_team_count[era]
    p_era = 1/7
    p_combo = p_era / n
    print(f"{era:<8} {n:>6} {p_era:>8.4f} {p_combo:>20.5f}")

print(f"\nTotal valid combos: {len(valid_combos)}")
print(f"Total players: {len(all_players):,}")

---
## 2. The SPG/BPG Adjustment — `adjustSpgBpg`

Old-era players (1960s, some 1970s) have `null` for steals and blocks — the data wasn't recorded.  
The engine compensates by scaling up the stats of players who **do** have recorded values.

**Original JS (from Engine.js chunk 89050):**
```javascript
function adjustSpgBpg(players) {
    const spgVals = players.filter(p => p.spg > 0).map(p => p.spg);
    const bpgVals = players.filter(p => p.bpg > 0).map(p => p.bpg);
    return {
        adjustedSpg: spgVals.reduce((a,b) => a+b, 0) * (spgVals.length > 0 ? 5/spgVals.length : 1),
        adjustedBpg: bpgVals.reduce((a,b) => a+b, 0) * (bpgVals.length > 0 ? 5/bpgVals.length : 1)
    };
}
```

In [ ]:
js_adjust = '''
function adjustSpgBpg(players) {
    const spgVals = players.filter(p => p.spg > 0).map(p => p.spg);
    const bpgVals = players.filter(p => p.bpg > 0).map(p => p.bpg);
    return {
        adjustedSpg: spgVals.reduce((a,b)=>a+b,0) * (spgVals.length>0 ? 5/spgVals.length : 1),
        adjustedBpg: bpgVals.reduce((a,b)=>a+b,0) * (bpgVals.length>0 ? 5/bpgVals.length : 1)
    };
}
'''

# Test: Wilt (no spg/bpg) + 4 modern players with spg=2.0, bpg=1.5
test_roster = [
    {'spg': None, 'bpg': None},   # Wilt - 1960s, no data
    {'spg': 2.0,  'bpg': 1.5},
    {'spg': 2.0,  'bpg': 1.5},
    {'spg': 2.0,  'bpg': 1.5},
    {'spg': 2.0,  'bpg': 1.5},
]

js_code = js_adjust + f'''
const roster = {json.dumps(test_roster)};
const result = adjustSpgBpg(roster);
console.log(JSON.stringify(result));
'''

js_result = json.loads(run_js(js_code))
py_result = adjust_spg_bpg(test_roster)

print('adjustSpgBpg — Wilt (null) + 4 players with spg=2.0, bpg=1.5:')
print(f'  JS:     adjustedSpg={js_result["adjustedSpg"]:.4f}  adjustedBpg={js_result["adjustedBpg"]:.4f}')
print(f'  Python: adjustedSpg={py_result["adjustedSpg"]:.4f}  adjustedBpg={py_result["adjustedBpg"]:.4f}')
print(f'  Match: {abs(js_result["adjustedSpg"] - py_result["adjustedSpg"]) < 1e-9}')
print()
print(f'Scaling factor: {js_result["adjustedSpg"] / (4*2.0):.3f}x  (4 valid players, scale = 5/4 = 1.25x)')
print('The 4 modern players carry defensive credit for all 5 slots.')

---
## 3. Team OVR Formula — `calculateTeamResult`

The team strength rating is a weighted sum of cumulative stats versus benchmark totals.

**Original JS:**
```javascript
// Benchmark denominators and weights
const s=133.4, o=39.7, d=29.3, p=6.1, c=3.2;  // PPG, RPG, APG, SPG, BPG
const a=0.46,  r=0.25, n=0.18, i=0.07, l=0.04; // weights

const teamOvr = Math.round(100 * (
    sumPpg/s*a + sumRpg/o*r + sumApg/d*n + adjSpg/p*i + adjBpg/c*l
) * 10) / 10;

const wins = Math.round(82 * Math.pow(Math.min(teamOvr/110, 1), 1.15));
```

In [ ]:
js_team = '''
function adjustSpgBpg(players) {
    const spgVals = players.filter(p => p.spg > 0).map(p => p.spg);
    const bpgVals = players.filter(p => p.bpg > 0).map(p => p.bpg);
    return {
        adjustedSpg: spgVals.reduce((a,b)=>a+b,0) * (spgVals.length>0 ? 5/spgVals.length : 1),
        adjustedBpg: bpgVals.reduce((a,b)=>a+b,0) * (bpgVals.length>0 ? 5/bpgVals.length : 1)
    };
}
function calculateTeamResult(players) {
    const {adjustedSpg, adjustedBpg} = adjustSpgBpg(players);
    const sumPpg = players.reduce((s,p) => s+(p.ppg||0), 0);
    const sumRpg = players.reduce((s,p) => s+(p.rpg||0), 0);
    const sumApg = players.reduce((s,p) => s+(p.apg||0), 0);
    const teamOvr = Math.round(100*(
        sumPpg/133.4*0.46 + sumRpg/39.7*0.25 + sumApg/29.3*0.18 +
        adjustedSpg/6.1*0.07 + adjustedBpg/3.2*0.04
    )*10)/10;
    const wins = Math.round(82 * Math.pow(Math.min(teamOvr/110, 1), 1.15));
    return {teamOvr, wins};
}
'''

# Test with a real lineup: Wilt + MJ + Giannis + Oscar + McAdoo
test_lineup = [
    {'player':'Wilt Chamberlain','ppg':41.46,'rpg':25.1,'apg':3.02,'spg':None,'bpg':None},
    {'player':'Michael Jordan',  'ppg':32.65,'rpg':6.15,'apg':5.93,'spg':2.81,'bpg':1.18},
    {'player':'Giannis',         'ppg':29.6, 'rpg':11.3,'apg':6.0, 'spg':1.0, 'bpg':1.1},
    {'player':'Oscar Robertson', 'ppg':29.66,'rpg':8.73,'apg':10.5,'spg':None,'bpg':None},
    {'player':'Bob McAdoo',      'ppg':28.24,'rpg':12.67,'apg':2.59,'spg':1.14,'bpg':2.42},
]

js_code = js_team + f'''
const lineup = {json.dumps(test_lineup)};
console.log(JSON.stringify(calculateTeamResult(lineup)));
'''

js_result = json.loads(run_js(js_code))
py_result = calculate_team_result(test_lineup)

print('Test lineup: Wilt + MJ + Giannis + Oscar Robertson + McAdoo')
print(f'  JS:     OVR={js_result["teamOvr"]}  wins={js_result["wins"]}')
print(f'  Python: OVR={py_result["teamOvr"]}  wins={py_result["wins"]}')
print(f'  Match: {js_result == {"teamOvr": py_result["teamOvr"], "wins": py_result["wins"]}}')

---
## 4. The Win Projection Curve

The relationship between OVR and wins is **non-linear** — each additional win near 82 is exponentially harder.

In [ ]:
js_wins = '''
function projectedWins(ovr) {
    return Math.round(82 * Math.pow(Math.min(ovr/110, 1), 1.15));
}
const results = [];
for (let ovr = 60; ovr <= 115; ovr += 5) {
    results.push({ovr, wins: projectedWins(ovr)});
}
console.log(JSON.stringify(results));
'''

curve = json.loads(run_js(js_wins))
py_curve = [(r['ovr'], projected_wins(r['ovr'])) for r in curve]

print(f"{'OVR':>5}  {'JS wins':>8}  {'Py wins':>8}  {'Match':>6}")
print('-' * 35)
for r, (_, py_w) in zip(curve, py_curve):
    match = '✓' if r['wins'] == py_w else '✗'
    threshold = ' ← 82-0 threshold' if r['wins'] == 82 and r['ovr'] <= 115 and r['ovr'] >= 105 else ''
    print(f"{r['ovr']:>5}  {r['wins']:>8}  {py_w:>8}  {match:>6}{threshold}")

---
## 5. Era Ceilings — How Players Are Rated Relative to Their Era

The individual player rating (used in HoopIQ mode) normalises stats against era-specific peaks.  
Classic mode uses raw absolute stats — but understanding ceilings explains why old-era big men dominate.

In [ ]:
from engine import ERA_CEILINGS

print(f"{'Era':<8} {'PPG ceil':>10} {'RPG ceil':>10} {'APG ceil':>10} {'SPG ceil':>10} {'BPG ceil':>10}")
print('-' * 58)
for era in ['1960s','1970s','1980s','1990s','2000s','2010s','2020s']:
    c = ERA_CEILINGS[era]
    print(f"{era:<8} {c['ppg']:>10} {c['rpg']:>10} {c['apg']:>10} {c['spg']:>10} {c['bpg']:>10}")

print()
print('Wilt GSW 1960s: 41.46 PPG vs 30.0 ceiling = 138% of era peak')
print('A 2020s player averaging 30 PPG = 107% of era peak')
print('In CLASSIC mode both count at face value — Wilt still wins on raw numbers.')

---
## 6. Player Ranking — `stat_contribution`

The optimizer scores each player by their direct per-player contribution to team OVR.  
This is the stat_contribution function used to rank players and make reroll decisions.

In [ ]:
js_sc = '''
function statContribution(p) {
    return (
        (p.ppg||0)/133.4*0.46 +
        (p.rpg||0)/39.7*0.25  +
        (p.apg||0)/29.3*0.18  +
        (p.spg||0)/6.1*0.07   +
        (p.bpg||0)/3.2*0.04
    );
}
'''

test_players = [
    {'name':'Wilt GSW 1960s', 'ppg':41.46,'rpg':25.1,'apg':3.02,'spg':0,'bpg':0},
    {'name':'Kareem MIL 1970s','ppg':30.43,'rpg':15.32,'apg':4.31,'spg':1.22,'bpg':3.41},
    {'name':'Jokic DEN 2020s', 'ppg':26.9,'rpg':12.4,'apg':9.3,'spg':1.4,'bpg':0.8},
    {'name':'MJ CHI 1980s',    'ppg':32.65,'rpg':6.15,'apg':5.93,'spg':2.81,'bpg':1.18},
    {'name':'Role player',     'ppg':8.0,  'rpg':3.0, 'apg':1.5,'spg':0.5,'bpg':0.2},
]

js_code = js_sc + '\n'.join(
    f'console.log(JSON.stringify({{name:"{p["name"]}", sc: statContribution({json.dumps({k:v for k,v in p.items() if k!="name"})})}}));'
    for p in test_players
)

js_lines = run_js(js_code).split('\n')
print(f"{'Player':<22} {'JS sc':>8}  {'Py sc':>8}  {'Mental':>8}  {'Tier':>6}")
print('-' * 60)
for line, p in zip(js_lines, test_players):
    js_val = json.loads(line)['sc']
    py_player = {k:v for k,v in p.items() if k != 'name'}
    py_player['spg'] = py_player['spg'] or None  # restore null for Wilt
    py_val = stat_contribution(p)
    mental = p['ppg'] + 2*p['rpg'] + 2*p['apg'] + 3*(p['spg'] or 0) + 3*(p['bpg'] or 0)
    tier = 'S' if mental>=80 else 'A' if mental>=63 else 'B' if mental>=50 else 'C' if mental>=46 else 'Reroll'
    print(f"{p['name']:<22} {js_val:>8.4f}  {py_val:>8.4f}  {mental:>8.1f}  {tier:>6}")

---
## 7. Best Team Per Era

Which team/era combos have the highest-value player available?

In [ ]:
from collections import defaultdict
from optimizer import _player_value

ctx = build_market_context(teams, valid_combos)

def mental(p):
    return (p.get('ppg') or 0) + 2*(p.get('rpg') or 0) + 2*(p.get('apg') or 0) + 3*(p.get('spg') or 0) + 3*(p.get('bpg') or 0)

for era in ['1960s','1970s','1980s','1990s','2000s','2010s','2020s']:
    combos = [(team, ctx['combo_players'][(team, era)]) for team in ctx['era_to_teams'].get(era, ())]
    combos = [(team, players) for team, players in combos if players]
    combos.sort(key=lambda x: mental(x[1][0]), reverse=True)
    print(f'  {era}  — top 4:')
    for team, players in combos[:4]:
        best = players[0]
        ms = mental(best)
        tier = 'S' if ms>=80 else 'A' if ms>=63 else 'B' if ms>=50 else 'C'
        print(f'    [{tier}] {ms:5.1f}  {team:4s}  {best["player"]}  ({best.get("ppg")}/{best.get("rpg")}/{best.get("apg")})')
    print()

---
## 8. The 82-0 Threshold

What combined mental score do 5 players need to reach 82 wins?

In [ ]:
import itertools
from simulator import can_player_play_position
from engine import POSITIONS

# Find the minimum-mental-score valid roster that hits 82 wins
# Sample from top 35 unique players
seen_names = {}
for p in sorted(all_players, key=mental, reverse=True):
    name = p['player']
    if name not in seen_names:
        seen_names[name] = p
    if len(seen_names) >= 35:
        break
top35 = list(seen_names.values())

min_82 = None
for combo in itertools.combinations(top35, 5):
    assigned = {}
    for pos in POSITIONS:
        for p in combo:
            if p not in assigned.values() and can_player_play_position(p, pos):
                assigned[pos] = p
                break
    if len(assigned) < 5:
        continue
    result = calculate_team_result(list(assigned.values()))
    if result['wins'] >= 82:
        total = sum(mental(p) for p in assigned.values())
        if min_82 is None or total < min_82[0]:
            min_82 = (total, result, assigned)

total, result, assigned = min_82
print(f'Minimum mental score total for 82-0: {total:.1f}')
print(f'  OVR={result["teamOvr"]}  wins={result["wins"]}')
print()
print(f'  Lineup:')
for pos, p in assigned.items():
    print(f'  {pos}: {p["player"]:30s} {p["team"]} {p["decade"]}  mental={mental(p):.1f}')

print()
print('Rule of thumb: aim for combined mental score ≥ 320')
print('  (provides buffer for APG-heavy teams where assists are slightly overweighted in the formula)')

---
## 9. SPG/BPG Scaling Bonus — The 1960s Pairing Advantage

In [ ]:
print('Effect of pairing old-era (null SPG/BPG) players with modern defenders:\n')

modern_defender = {'ppg':18,'rpg':8,'apg':2,'spg':2.5,'bpg':3.0}

cases = [
    ('0 null players (all modern)',  [modern_defender]*5),
    ('1 null player (e.g. Wilt)',     [{'ppg':41,'rpg':25,'apg':3,'spg':None,'bpg':None}] + [modern_defender]*4),
    ('2 null players',               [{'ppg':30,'rpg':20,'apg':4,'spg':None,'bpg':None}]*2 + [modern_defender]*3),
    ('3 null players',               [{'ppg':28,'rpg':18,'apg':4,'spg':None,'bpg':None}]*3 + [modern_defender]*2),
]

for label, roster in cases:
    adj = adjust_spg_bpg(roster)
    n_valid = sum(1 for p in roster if p.get('spg') and p['spg'] > 0)
    scale = 5/n_valid if n_valid > 0 else 1
    result = calculate_team_result(roster)
    print(f'{label}:')
    print(f'  Valid SPG players: {n_valid}  → scale factor {scale:.2f}x')
    print(f'  adj_SPG={adj["adjustedSpg"]:.2f}  adj_BPG={adj["adjustedBpg"]:.2f}  OVR={result["teamOvr"]}  wins={result["wins"]}')
    print()

---
## 10. Reroll Decision Logic

The optimizer computes the expected value of each spin using precomputed baselines.  
Here we show what the expected best player per position is across a random spin.

In [ ]:
from engine import POSITIONS

print('Expected best player contribution per open slot (baseline for reroll decisions):')
print()
print(f"  {'Slot':<5} {'Expected sc':>12}  {'Mental equiv':>14}  {'Interpretation'}")
print('  ' + '-'*60)
for pos in POSITIONS:
    baseline = ctx['expected_pos_baseline'][pos]
    mental_equiv = baseline * 290
    print(f"  {pos:<5} {baseline:>12.4f}  {mental_equiv:>14.1f}  {'reroll if best pick below this' if pos == 'PG' else ''}")

print()
print('If the best available player for your open slot has sc < baseline → reroll is +EV')
print('The reroll threshold in mental score terms is roughly 46-50 (Tier C/D boundary)')